# Hybrid Retriever - Combining Dense and Sparse Retriever

In [1]:
import os
from dotenv import load_dotenv

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

load_dotenv(override=True)

True

In [2]:
# Step 1: Sample documents

docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="LangChain can be used to develop agentic AI application."),
    Document(page_content="LangChain has many types of retrievers"),
]

# Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model_name = os.getenv("HUGGING_FACE_EMBEDDING_MODEL"))
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11859.37it/s]


In [4]:
### Step 3: Sparse Reteriver (BM25 Algorithm)
sparse_reteriver = BM25Retriever.from_documents(docs)
sparse_reteriver.k = 3 ## Top k document from retriever



In [5]:
# Step 4: Combine with Ensemble Retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_reteriver],
    weight=[0.7, 0.3]
)

In [6]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x130fb23c0>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x12e331e80>, k=3)], weights=[0.5, 0.5])

In [8]:
# Step 5: Query and get Results
query = "How can i build an application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6: Print Results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}: \n {doc.page_content}")


🔹 Document 1: 
 LangChain helps build LLM applications.

🔹 Document 2: 
 LangChain can be used to develop agentic AI application.

🔹 Document 3: 
 LangChain has many types of retrievers

🔹 Document 4: 
 Pinecone is a vector database for semantic search.


# RAG Pipeline with hybrid retriever

In [9]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [10]:
# Step 5: Prompt template
prompt = PromptTemplate.from_template(
    """Answer the question based on the context below.
    
    Context: {context}
    
    Qusetion: {input}
    
    Answer: """
)

# Step 6: LLM
llm = init_chat_model(os.getenv("OLLAMA_CHAT_MODEL"))

llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')

In [11]:
# Create stuff document chain
document_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)

# Create full RAG chain
rag_chain = create_retrieval_chain(retriever=hybrid_retriever, combine_docs_chain=document_chain)

rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x130fb23c0>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x12e331e80>, k=3)], weights=[0.5, 0.5]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='Answer the question based on the context below.\n\n    Context: {context}\n\n    Qusetion: {input}\n\n    Answer: ')
            | ChatOllama(me

In [12]:
# Step 9: Ask a question
query = {"input": "How can i build an app using LLMs?"}
response = rag_chain.invoke(query)

# Step 10: output
print(f"✅ Answer: \n", response['answer'])

print("\n📄 Source Documents:")
for i, doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

✅ Answer: 
 You can use LangChain, which helps build LLM applications. Specifically, LangChain can be used to develop agentic AI applications and offers many types of retrievers. (Additionally, you may use tools like Pinecone, a vector database for semantic search.)

📄 Source Documents:

Doc 1: LangChain helps build LLM applications.

Doc 2: LangChain can be used to develop agentic AI application.

Doc 3: LangChain has many types of retrievers

Doc 4: Pinecone is a vector database for semantic search.
